# All-In-One PCA Notebook (df_stim + df_units + NWB)

This notebook is built to run PCA from exactly three experiment outputs:

1. `df_stim.json`
2. `df_units.json`
3. associated NWB file

No other experiment-specific files are required.


## What It Does

- Loads and validates all 3 files.
- Infers trial/block labels (`baseline`, `opto_epoch_k`, `washout_epoch_k`) from opto TTL state in `df_stim`.
- Loads unit metadata from `df_units` (memory-safe for large files).
- Loads spike times from NWB units table.
- Aligns spikes to event times, bins into firing rate, and runs:
  - trial-level PCA
  - trajectory PCA by block label
  - optional region-level PCA
- Saves outputs to `extra_files`.


## Conda Environment Setup (PCA)

Create and use a dedicated environment before running this notebook:

```bash
conda create -n pca_neuropixels python=3.11 -y
conda activate pca_neuropixels
pip install numpy pandas matplotlib seaborn scikit-learn scipy pynwb ijson jupyterlab
```

Then launch Jupyter:

```bash
jupyter lab
```

Optional kernel registration:

```bash
python -m ipykernel install --user --name pca_neuropixels --display-name "Python (pca_neuropixels)"
```


### Cell 1: Import Core Libraries

**What this cell does**
- Imports analysis libraries (`numpy`, `pandas`, `matplotlib`, `seaborn`, `scikit-learn`).
- Sets plotting style and suppresses noisy warnings.

**How to use it**
- Run this first.
- If imports fail, install missing packages before continuing.


In [25]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

sns.set_context("talk")
sns.set_style("white")
warnings.filterwarnings("ignore")


### Cell 2: Set the 3 Required Input Paths and Analysis Settings

**What this cell does**
- Sets the 3 required input files (`df_stim`, `df_units`, NWB).
- Sets analysis window and PCA parameters.
- Adds filter controls for `PROBE_FILTER` and `KSLABEL_FILTER`.

**How to use it**
- Set `PROBE_FILTER='A'` (or B-F) to keep one probe.
- Set `KSLABEL_FILTER='good'` or `2` for good units only.
- Set `KSLABEL_FILTER='mua'` or `1` for MUA only.
- Leave filters as `None` to keep all units.


In [26]:
# -----------------------------
# USER CONFIG (ONLY 3 FILE PATHS)
# -----------------------------

DF_STIM_PATH = Path(r"h:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00\Record Node 103\experiment1\recording1\continuous\intermediates\df_stim.json")
DF_UNITS_PATH = Path(r"h:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00\Record Node 103\experiment1\recording1\continuous\intermediates\df_units.json")
NWB_INPUT_PATH = Path(r"g:\Grant\neuropixels\nwb\Reach15")

WINDOW_START_S = -0.5
WINDOW_END_S = 3.5
BIN_SIZE_S = 0.05

N_COMPONENTS = 12
SMOOTH_SIGMA = 2

# Unit filters
PROBE_FILTER = None        # 'A'..'F' or None
KSLABEL_FILTER = None      # 'good'/'mua' or 2/1 or None
UNIT_FILTER_QUERY = None   # e.g. "quality == 'good'"
MAX_UNITS = None

REQUIRE_DF_UNITS = True
DF_UNITS_SELECTED_COLUMNS = [
  "cluster_id", "quality", "peak_channel", "location", "brain_region",
  "firing_rate", "probe", "KSLabel", "kslabel"
]

OUT_DIR = Path.cwd() / "extra_files"
OUT_DIR.mkdir(parents=True, exist_ok=True)


### Debug Cell A: Inspect `df_units.json` Top-Level Keys

- Streams `df_units.json` with `ijson` and lists top-level keys.
- Use this to confirm exact probe/KSLabel key names.


In [ ]:
import ijson

top_keys = []
with open(DF_UNITS_PATH, "rb") as f:
    for prefix, event, value in ijson.parse(f):
        if prefix == "" and event == "map_key":
            top_keys.append(value)

print("n_top_keys:", len(top_keys))
print("example keys:", top_keys[:40])
print("probe-like keys:", [k for k in top_keys if "probe" in k.lower()])
print("ks/label-like keys:", [k for k in top_keys if ("ks" in k.lower() or "label" in k.lower())])


### Debug Cell B: Load Minimal Unit View and Test Probe/KSLabel Filter

- Loads only minimal columns (`cluster_id`, probe key, KS label key).
- Prints value counts and filtered preview.


In [ ]:
import numpy as np
import pandas as pd
import ijson


def _load_units_cols(path, cols):
    data = {c: {} for c in cols}
    with open(path, "rb") as f:
        current_top = None
        for prefix, event, value in ijson.parse(f):
            if prefix == "" and event == "map_key":
                current_top = value
                continue
            if current_top in data and event in {"string", "number", "boolean", "null"}:
                if prefix.startswith(current_top + "."):
                    row_key = prefix.split(".", 1)[1]
                    data[current_top][row_key] = value
    rows = sorted({k for c in cols for k in data[c].keys()}, key=lambda x: int(x) if str(x).isdigit() else str(x))
    out = pd.DataFrame(index=rows)
    for c in cols:
        out[c] = pd.Series(data[c]).reindex(rows).values
    return out.reset_index(drop=True)


def _norm_ks(v):
    if pd.isna(v):
        return np.nan
    try:
        iv = int(float(v))
        if iv == 2:
            return "good"
        if iv == 1:
            return "mua"
    except Exception:
        pass
    s = str(v).strip().lower()
    if s in {"good", "2"}:
        return "good"
    if s in {"mua", "1"}:
        return "mua"
    return s

probe_key = next((k for k in top_keys if k.lower() == "probe"), None) or next((k for k in top_keys if "probe" in k.lower()), None)
ks_key = next((k for k in top_keys if k in {"KSLabel", "kslabel", "ks_label"}), None) or next((k for k in top_keys if ("ks" in k.lower() or "label" in k.lower())), None)

cols = ["cluster_id"]
if probe_key is not None:
    cols.append(probe_key)
if ks_key is not None:
    cols.append(ks_key)

units_view = _load_units_cols(DF_UNITS_PATH, cols)
print("loaded columns:", list(units_view.columns))
print(units_view.head())

if probe_key is not None:
    print("\nProbe counts:")
    print(units_view[probe_key].astype(str).str.upper().str[0].value_counts(dropna=False))
if ks_key is not None:
    print("\nKSLabel counts:")
    print(units_view[ks_key].map(_norm_ks).value_counts(dropna=False))

DEBUG_PROBE = "A"
DEBUG_KS = "good"

u = units_view.copy()
u["_probe_norm"] = u[probe_key].astype(str).str.strip().str.upper().str[0] if probe_key is not None else np.nan
u["_ks_norm"] = u[ks_key].map(_norm_ks) if ks_key is not None else np.nan
u_filt = u[(u["_probe_norm"] == DEBUG_PROBE) & (u["_ks_norm"] == DEBUG_KS)]

print(f"\nFiltered preview (probe={DEBUG_PROBE}, ks={DEBUG_KS}) -> n={len(u_filt)}")
print(u_filt.head())


### Cell 3: Define Helper Functions (Validation, Labeling, Utilities)

**What this cell does**
- Provides reusable helpers for:
  - file checks
  - NWB path resolution
  - opto-state inference from `df_stim`
  - block labeling (`baseline`, `opto_epoch_k`, `washout_epoch_k`)
  - z-scoring and region-column detection

**How to use it**
- Run once after config.
- No edits needed unless your schema changes significantly.


In [19]:
def zscore_rows(x: np.ndarray) -> np.ndarray:
    scaler = StandardScaler(with_mean=True, with_std=True)
    return scaler.fit_transform(x.T).T


def verify_file(path: Path, label: str):
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")
    size_gb = path.stat().st_size / (1024 ** 3)
    print(f"{label}: {path} ({size_gb:.3f} GB)")


def resolve_nwb_path(nwb_input: Path) -> Path:
    """Accepts direct NWB file path or directory containing one .nwb/HDF5 file."""
    if not nwb_input.exists():
        raise FileNotFoundError(f"NWB input path not found: {nwb_input}")

    if nwb_input.is_file():
        return nwb_input

    # Directory case
    nwb_candidates = sorted(list(nwb_input.rglob("*.nwb")))
    if len(nwb_candidates) == 1:
        return nwb_candidates[0]
    if len(nwb_candidates) > 1:
        raise ValueError(f"Multiple .nwb files found under {nwb_input}; please set NWB_INPUT_PATH to one file.")

    # Fallback: if directory has one file only, assume it is NWB/HDF5
    files = [p for p in nwb_input.iterdir() if p.is_file()]
    if len(files) == 1:
        return files[0]

    raise ValueError(f"Could not uniquely resolve NWB file from {nwb_input}")


def infer_is_opto(df_stim: pd.DataFrame) -> pd.Series:
    if "optogenetics_LED_state" in df_stim.columns:
        c = df_stim["optogenetics_LED_state"]
        if pd.api.types.is_numeric_dtype(c):
            return pd.to_numeric(c, errors="coerce").fillna(0) > 0
        return c.astype(str).str.lower().isin(["1", "true", "on", "high"])

    if "stimulus" in df_stim.columns:
        s = df_stim["stimulus"].astype(str).str.lower()
        return s.str.contains("opto|laser|led|stim", regex=True)

    for c in df_stim.columns:
        cl = c.lower()
        if "opto" in cl or "laser" in cl or "led" in cl:
            v = df_stim[c]
            if pd.api.types.is_numeric_dtype(v):
                return pd.to_numeric(v, errors="coerce").fillna(0) > 0
            return v.astype(str).str.lower().isin(["1", "true", "on", "high"])

    raise ValueError("Unable to infer opto state from df_stim.json")


def label_blocks_from_opto(is_opto: pd.Series) -> pd.DataFrame:
    is_opto = is_opto.astype(bool).reset_index(drop=True)
    block_id = (is_opto != is_opto.shift(1, fill_value=is_opto.iloc[0])).cumsum()

    mapping = {}
    seen_opto = False
    opto_epoch = 1
    wash_epoch = 1

    for b in block_id.unique():
        state = bool(is_opto[block_id == b].iloc[0])
        if state:
            mapping[b] = f"opto_epoch_{opto_epoch}"
            opto_epoch += 1
            seen_opto = True
        else:
            if not seen_opto:
                mapping[b] = "baseline"
            else:
                mapping[b] = f"washout_epoch_{wash_epoch}"
                wash_epoch += 1

    return pd.DataFrame({
        "block_id": block_id,
        "is_opto": is_opto,
        "block_label": block_id.map(mapping)
    })


def pick_region_column(df: pd.DataFrame):
    for c in ["brain_region", "location", "region", "acronym", "structure", "ccf_acronym"]:
        if c in df.columns:
            return c
    return None


### Cell 4: Validate Inputs and Resolve NWB File

**What this cell does**
- Resolves the NWB file when `NWB_INPUT_PATH` is a directory.
- Verifies all three required files exist and reports sizes.

**How to use it**
- Run to confirm your paths are correct before heavy loading.
- Fix any path errors before proceeding.


In [20]:
# -----------------------------
# USER CONFIG (ONLY 3 FILE PATHS)
# -----------------------------

DF_STIM_PATH = Path(r"h:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00\Record Node 103\experiment1\recording1\continuous\intermediates\df_stim.json")
DF_UNITS_PATH = Path(r"h:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00\Record Node 103\experiment1\recording1\continuous\intermediates\df_units.json")
NWB_INPUT_PATH = Path(r"g:\Grant\neuropixels\nwb\Reach15")

WINDOW_START_S = -0.5
WINDOW_END_S = 3.5
BIN_SIZE_S = 0.05

N_COMPONENTS = 12
SMOOTH_SIGMA = 2

# Unit filters
PROBE_FILTER = None        # 'A'..'F' or None
KSLABEL_FILTER = None      # 'good'/'mua' or 2/1 or None
UNIT_FILTER_QUERY = None   # e.g. "quality == 'good'"
MAX_UNITS = None

REQUIRE_DF_UNITS = True
DF_UNITS_SELECTED_COLUMNS = [
  "cluster_id", "quality", "peak_channel", "location", "brain_region",
  "firing_rate", "probe", "KSLabel", "kslabel"
]

OUT_DIR = Path.cwd() / "extra_files"
OUT_DIR.mkdir(parents=True, exist_ok=True)


df_stim.json: h:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00\Record Node 103\experiment1\recording1\continuous\intermediates\df_stim.json (0.118 GB)
df_units.json: h:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00\Record Node 103\experiment1\recording1\continuous\intermediates\df_units.json (6.144 GB)
NWB file: g:\Grant\neuropixels\nwb\Reach15 (1.587 GB)


### Cell 5: Load and Prepare `df_stim.json`

**What this cell does**
- Loads stimulation/event table.
- Standardizes event timing into `event_time_s`.
- Infers opto ON/OFF and block labels.
- Creates trial index and prints block summary.

**How to use it**
- Run and confirm block counts match your experiment design.
- If labels look wrong, inspect TTL/state values in `df_stim`.


In [21]:
def load_stim_df(path: Path) -> pd.DataFrame:
    df = pd.read_json(path)

    if "start_time" in df.columns:
        event_time = pd.to_numeric(df["start_time"], errors="coerce")
    elif "time" in df.columns:
        event_time = pd.to_numeric(df["time"], errors="coerce")
    else:
        raise ValueError(f"df_stim missing time column (start_time/time). Columns: {list(df.columns)}")

    out = df.copy()
    out["event_time_s"] = event_time
    out = out.dropna(subset=["event_time_s"]).sort_values("event_time_s").reset_index(drop=True)

    blk = label_blocks_from_opto(infer_is_opto(out))
    out = pd.concat([out, blk], axis=1)
    out["trial_index"] = np.arange(len(out), dtype=int)
    return out


stim_df = load_stim_df(DF_STIM_PATH)
print("df_stim columns:", list(stim_df.columns))
print("n_events:", len(stim_df))
print(stim_df[["trial_index", "event_time_s", "is_opto", "block_id", "block_label"]].head(10))
print("\nBlock counts:")
print(stim_df["block_label"].value_counts())
stim_df


df_stim columns: ['start_time', 'stop_time', 'stimulus', 'optogenetics_LED_state', 'event_time_s', 'block_id', 'is_opto', 'block_label', 'trial_index']
n_events: 1352824
   trial_index  event_time_s  is_opto  block_id block_label
0            0  10028.508867    False         0    baseline
1            1  10028.515500    False         0    baseline
2            2  10028.522200    False         0    baseline
3            3  10028.528833    False         0    baseline
4            4  10028.535533    False         0    baseline
5            5  10028.542167    False         0    baseline
6            6  10028.548867    False         0    baseline
7            7  10028.555500    False         0    baseline
8            8  10028.562200    False         0    baseline
9            9  10028.568833    False         0    baseline

Block counts:
block_label
baseline             133718
washout_epoch_120    113492
washout_epoch_40     109426
washout_epoch_20     106777
washout_epoch_80     103191
   

,start_time,stop_time,stimulus,optogenetics_LED_state,event_time_s,block_id,is_opto,block_label,trial_index
0,10028.508867,10028.508867,frame_events_timestamp,0,10028.508867,0,False,baseline,0
1,10028.515500,10028.515500,frame_events_timestamp,0,10028.515500,0,False,baseline,1
2,10028.522200,10028.522200,frame_events_timestamp,0,10028.522200,0,False,baseline,2
3,10028.528833,10028.528833,frame_events_timestamp,0,10028.528833,0,False,baseline,3
4,10028.535533,10028.535533,frame_events_timestamp,0,10028.535533,0,False,baseline,4
...,...,...,...,...,...,...,...,...,...
1352819,19028.808167,19028.808167,frame_events_timestamp,0,19028.808167,380,False,washout_epoch_190,1352819
1352820,19028.814867,19028.814867,frame_events_timestamp,0,19028.814867,380,False,washout_epoch_190,1352820
1352821,19028.821500,19028.821500,frame_events_timestamp,0,19028.821500,380,False,washout_epoch_190,1352821
1352822,19028.828200,19028.828200,frame_events_timestamp,0,19028.828200,380,False,washout_epoch_190,1352822


### Cell 6: Load `df_units.json` (Memory-Safe)

**What this cell does**
- Loads selected columns from very large `df_units.json` using `ijson`.
- Requires `cluster_id` for downstream unit alignment.
- Fails fast when `REQUIRE_DF_UNITS=True` and loading fails.

**How to use it**
- Run as-is for large files.
- If `ijson` is missing, install it and re-run.
- Add/remove fields in `DF_UNITS_SELECTED_COLUMNS` as needed.


In [22]:
REQUIRE_DF_UNITS = False

In [23]:
def load_units_json_selected_columns(path: Path, selected_columns):
    try:
        import ijson
    except Exception as e:
        raise ImportError("ijson is required for large df_units loading. Install with: pip install ijson") from e

    wanted = [c for c in selected_columns if isinstance(c, str) and c]
    if not wanted:
        raise ValueError("DF_UNITS_SELECTED_COLUMNS is empty.")

    data = {c: {} for c in wanted}
    top_keys_seen = set()

    with open(path, "rb") as f:
        current_top = None
        for prefix, event, value in ijson.parse(f):
            if prefix == "" and event == "map_key":
                current_top = value
                top_keys_seen.add(value)
                continue

            if current_top in data and event in {"string", "number", "boolean", "null"}:
                if prefix.startswith(current_top + "."):
                    row_key = prefix.split(".", 1)[1]
                    data[current_top][row_key] = value

    available = [c for c in wanted if c in top_keys_seen]
    if "cluster_id" not in top_keys_seen:
        raise ValueError("df_units.json missing required top-level key: cluster_id")

    if len(available) == 0:
        raise ValueError("None of selected df_units columns were found.")

    rows = sorted({k for c in available for k in data[c].keys()}, key=lambda x: int(x) if str(x).isdigit() else str(x))
    out = pd.DataFrame(index=rows)
    for c in available:
        out[c] = pd.Series(data[c]).reindex(rows).values
    out = out.reset_index(drop=True)

    out["cluster_id"] = pd.to_numeric(out["cluster_id"], errors="coerce")
    out = out.dropna(subset=["cluster_id"]).reset_index(drop=True)
    out["cluster_id"] = out["cluster_id"].astype(int)
    return out


def load_units_df(path: Path) -> pd.DataFrame:
    # For very large files this selective path is the intended mode.
    return load_units_json_selected_columns(path, DF_UNITS_SELECTED_COLUMNS)


try:
    units_df = load_units_df(DF_UNITS_PATH)
    print("Loaded df_units columns:", list(units_df.columns), "rows=", len(units_df))
except Exception as e:
    if REQUIRE_DF_UNITS:
        raise RuntimeError(f"Failed to load df_units.json: {e}")
    print("Proceeding without df_units due to load issue:", e)
    units_df = pd.DataFrame()


Loaded df_units columns: ['cluster_id'] rows= 4600


In [24]:
units_df

,cluster_id
0,0
1,1
2,2
3,3
4,4
...,...
4595,686
4596,687
4597,688
4598,689


### Cell 7: Load NWB Units and Spike Times

**What this cell does**
- Opens NWB and reads units table.
- Extracts `unit_id`, available unit metadata, and per-unit spike times.

**How to use it**
- Run after `df_units` load.
- If this fails, verify NWB integrity and `pynwb` installation.


In [11]:
def load_nwb_spikes_and_meta(nwb_path: Path):
    try:
        from pynwb import NWBHDF5IO
    except Exception as e:
        raise ImportError("pynwb is required. Install with: pip install pynwb") from e

    io = NWBHDF5IO(str(nwb_path), mode="r", load_namespaces=True)
    nwb = io.read()

    if nwb.units is None:
        raise ValueError("NWB units table not found.")

    ut = nwb.units
    n_units = len(ut.id[:])
    unit_ids = np.asarray(ut.id[:]).astype(int)

    meta = pd.DataFrame({"unit_id": unit_ids})
    candidate_cols = ["quality","peak_channel","location","brain_region","electrodes","probe","probe_name","KSLabel","kslabel","label"]
    for c in candidate_cols:
        if c in ut.colnames:
            try:
                meta[c] = list(ut[c][:])
            except Exception:
                pass

    spike_times = [np.asarray(ut["spike_times"][i], dtype=float) for i in range(n_units)]
    return io, nwb, meta, spike_times


io, nwb, nwb_units_meta, spike_times_by_unit = load_nwb_spikes_and_meta(NWB_PATH)
print("NWB units:", len(spike_times_by_unit))
print("NWB metadata columns:", list(nwb_units_meta.columns))


NWB units: 4600
NWB metadata columns: ['unit_id']


In [15]:
print('shape of spike_times_by_unit:', len(spike_times_by_unit), 'units; example unit 0 has', len(spike_times_by_unit[0]), 'spikes')


shape of spike_times_by_unit: 4600 units; example unit 0 has 14579 spikes


### Cell 8: Merge Unit Metadata and Apply Filters

**What this cell does**
- Merges NWB units with `df_units` metadata.
- Uses ID overlap when possible; otherwise uses positional fallback.
- Applies optional `UNIT_FILTER_QUERY` and `MAX_UNITS`.

**How to use it**
- Check printed overlap percentage.
- If overlap is unexpectedly low, verify `cluster_id` conventions in your pipeline.


In [16]:
def normalize_kslabel(v):
    if pd.isna(v):
        return np.nan
    try:
        iv = int(float(v))
        if iv == 2:
            return "good"
        if iv == 1:
            return "mua"
    except Exception:
        pass
    s = str(v).strip().lower()
    if s in {"2", "good", "single", "singleunit", "single_unit"}:
        return "good"
    if s in {"1", "mua", "multi", "multiunit", "multi_unit"}:
        return "mua"
    return s


def find_probe_col(df: pd.DataFrame):
    for c in ["probe", "probe_name", "probe_id", "probe_letter", "electrode_group"]:
        if c in df.columns:
            return c
    return None


def find_kslabel_col(df: pd.DataFrame):
    for c in ["KSLabel", "kslabel", "ks_label", "label", "quality"]:
        if c in df.columns:
            return c
    return None


unit_meta = nwb_units_meta.copy()

if not units_df.empty:
    overlap = set(unit_meta["unit_id"].astype(int)).intersection(set(units_df["cluster_id"].astype(int)))
    overlap_ratio = len(overlap) / max(1, len(unit_meta))
    print(f"unit_id/cluster_id overlap: {len(overlap)} ({overlap_ratio:.2%} of NWB units)")
    if overlap_ratio > 0.2:
        unit_meta = unit_meta.merge(units_df, left_on="unit_id", right_on="cluster_id", how="left", suffixes=("", "_dfunits"))
    else:
        print("Low ID overlap; using positional fallback merge for df_units metadata.")
        m = min(len(unit_meta), len(units_df))
        merged = unit_meta.iloc[:m].copy().reset_index(drop=True)
        for c in units_df.columns:
            if c != "cluster_id":
                merged[c] = units_df.iloc[:m][c].values
        unit_meta = merged
        spike_times_by_unit = spike_times_by_unit[:m]

probe_col = find_probe_col(unit_meta)
if PROBE_FILTER is not None:
    if probe_col is None:
        raise ValueError("PROBE_FILTER set but no probe column found.")
    target_probe = str(PROBE_FILTER).strip().upper()
    probe_norm = unit_meta[probe_col].astype(str).str.strip().str.upper().str[0]
    keep_idx = np.where(probe_norm == target_probe)[0]
    unit_meta = unit_meta.iloc[keep_idx].reset_index(drop=True)
    spike_times_by_unit = [spike_times_by_unit[i] for i in keep_idx]
    print(f"Applied PROBE_FILTER={target_probe}: kept {len(keep_idx)} units")
else:
    print("PROBE_FILTER=None: keeping all probes")

ks_col = find_kslabel_col(unit_meta)
if KSLABEL_FILTER is not None:
    if ks_col is None:
        raise ValueError("KSLABEL_FILTER set but no KS label column found.")
    target_ks = normalize_kslabel(KSLABEL_FILTER)
    if target_ks not in {"good", "mua"}:
        raise ValueError(f"KSLABEL_FILTER must map to good/mua; got: {KSLABEL_FILTER}")
    ks_norm = unit_meta[ks_col].map(normalize_kslabel)
    keep_idx = np.where(ks_norm == target_ks)[0]
    unit_meta = unit_meta.iloc[keep_idx].reset_index(drop=True)
    spike_times_by_unit = [spike_times_by_unit[i] for i in keep_idx]
    print(f"Applied KSLABEL_FILTER={target_ks}: kept {len(keep_idx)} units")
else:
    print("KSLABEL_FILTER=None: keeping all KS labels")

if UNIT_FILTER_QUERY:
    keep_idx = unit_meta.query(UNIT_FILTER_QUERY).index.values
    unit_meta = unit_meta.iloc[keep_idx].reset_index(drop=True)
    spike_times_by_unit = [spike_times_by_unit[i] for i in keep_idx]
    print(f"Applied UNIT_FILTER_QUERY: kept {len(keep_idx)} units")

if MAX_UNITS is not None:
    unit_meta = unit_meta.iloc[:MAX_UNITS].reset_index(drop=True)
    spike_times_by_unit = spike_times_by_unit[:MAX_UNITS]

if len(spike_times_by_unit) == 0:
    raise ValueError("No units left after filtering.")

print("Final units used:", len(spike_times_by_unit))
print("Merged unit metadata columns:", list(unit_meta.columns))
print(unit_meta.head())


unit_id/cluster_id overlap: 1316 (28.61% of NWB units)


UndefinedVariableError: name 'good' is not defined

### Cell 9: Bin Spikes Around Events

**What this cell does**
- Aligns each unitâ€™s spikes to each event time.
- Bins activity into firing rate (Hz) over the configured window.
- Produces `trials` array: `(n_trials, n_units, n_bins)`.

**How to use it**
- Run once inputs are loaded.
- Confirm output shape is reasonable for your session.


In [11]:
def bin_spikes_around_events(spike_times_list, event_times_s, win_start_s, win_end_s, bin_size_s):
    edges = np.arange(win_start_s, win_end_s + bin_size_s, bin_size_s)
    n_bins = len(edges) - 1
    n_trials = len(event_times_s)
    n_units = len(spike_times_list)

    X = np.zeros((n_trials, n_units, n_bins), dtype=np.float32)

    for u, st in enumerate(spike_times_list):
        st = np.asarray(st, dtype=float)
        if st.size == 0:
            continue
        for t, t0 in enumerate(event_times_s):
            left = t0 + win_start_s
            right = t0 + win_end_s
            i0 = np.searchsorted(st, left, side="left")
            i1 = np.searchsorted(st, right, side="right")
            rel = st[i0:i1] - t0
            if rel.size:
                counts, _ = np.histogram(rel, bins=edges)
                X[t, u, :] = counts / bin_size_s

    return X, edges[:-1]


event_times = stim_df["event_time_s"].to_numpy(dtype=float)
trials, time = bin_spikes_around_events(
    spike_times_list=spike_times_by_unit,
    event_times_s=event_times,
    win_start_s=WINDOW_START_S,
    win_end_s=WINDOW_END_S,
    bin_size_s=BIN_SIZE_S,
)

print("trials shape:", trials.shape, "(n_trials, n_units, n_bins)")


MemoryError: Unable to allocate 1.81 TiB for an array with shape (1352824, 4600, 80) and data type float32

### Cell 10: Trial-Level PCA

**What this cell does**
- Uses mean in-task activity per trial to create one feature vector per trial.
- Z-scores by unit, runs PCA, and plots trial clustering in PC subspaces.

**How to use it**
- Use this to compare how trial blocks separate in low-dimensional space.
- Check explained variance and visual separation by block label.


In [ ]:
# Trial-level PCA (one point per trial)
trial_type = stim_df["block_label"].to_numpy()
trial_types = pd.unique(trial_type)
t_type_ind = [np.where(trial_type == t)[0] for t in trial_types]

in_task = (time >= 0.0) & (time < (WINDOW_END_S - BIN_SIZE_S))
if in_task.sum() == 0:
    raise ValueError("No bins in in-task window. Check WINDOW_* settings.")

X_trial = np.vstack([trials[i, :, in_task].mean(axis=1) for i in range(trials.shape[0])]).T
X_trial_z = zscore_rows(X_trial)

pca_trial = PCA(n_components=min(N_COMPONENTS, X_trial_z.shape[0], X_trial_z.shape[1]))
Xp = pca_trial.fit_transform(X_trial_z.T).T
print("Trial PCA EVR (first 5):", np.round(pca_trial.explained_variance_ratio_[:5], 4))

projections = [(0, 1), (1, 2), (0, 2)]
pal = sns.color_palette("colorblind", len(trial_types))
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (i, j) in zip(axes, projections):
    for k, t in enumerate(trial_types):
        idx = t_type_ind[k]
        ax.scatter(Xp[i, idx], Xp[j, idx], s=30, alpha=0.8, color=pal[k], label=str(t))
    ax.set_xlabel(f"PC {i+1}")
    ax.set_ylabel(f"PC {j+1}")
axes[-1].legend(frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")
sns.despine()
plt.tight_layout()
plt.show()


### Cell 11: Trajectory PCA (Block-Averaged Time Courses)

**What this cell does**
- Averages trials within each block label across time.
- Runs PCA on concatenated block trajectories.
- Plots first 3 PCs over time with optional smoothing.

**How to use it**
- Use this to compare temporal neural dynamics across baseline/opto/washout blocks.
- Vertical dashed line marks event onset (`t=0`).


In [ ]:
# Trajectory PCA on trial-averaged time courses by block
trial_averages = []
kept_labels = []
for t, idx in zip(trial_types, t_type_ind):
    if len(idx) > 0:
        trial_averages.append(trials[idx].mean(axis=0))
        kept_labels.append(t)

if len(trial_averages) < 2:
    raise ValueError("Need >=2 non-empty block labels for trajectory PCA")

Xa = np.hstack(trial_averages)
Xa_z = zscore_rows(Xa)

pca_traj = PCA(n_components=min(N_COMPONENTS, Xa_z.shape[0], Xa_z.shape[1]))
Xa_p = pca_traj.fit_transform(Xa_z.T).T
print("Trajectory PCA EVR (first 5):", np.round(pca_traj.explained_variance_ratio_[:5], 4))

from scipy.ndimage import gaussian_filter1d

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharex=True)
n_bins = trials.shape[2]
for comp in range(min(3, Xa_p.shape[0])):
    ax = axes[comp]
    for k, lbl in enumerate(kept_labels):
        s = k * n_bins
        e = (k + 1) * n_bins
        x = Xa_p[comp, s:e]
        if SMOOTH_SIGMA and SMOOTH_SIGMA > 0:
            x = gaussian_filter1d(x, sigma=SMOOTH_SIGMA)
        ax.plot(time, x, lw=2, color=pal[k], label=str(lbl))
    ax.axvline(0, color="gray", ls="--", lw=1)
    ax.set_ylabel(f"PC {comp+1}")

axes[1].set_xlabel("Time from event (s)")
axes[-1].legend(frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")
sns.despine()
plt.tight_layout()
plt.show()


### Cell 12: Optional Region-Level PCA

**What this cell does**
- If region metadata exists, groups units by region.
- Builds region-averaged trial features and runs PCA.
- Skips automatically when region labels are unavailable/insufficient.

**How to use it**
- Useful for cross-region population structure.
- Ensure unit metadata includes reliable region annotations.


In [ ]:
# Optional: region-level PCA (if unit region metadata is available)
region_col = pick_region_column(unit_meta)
if region_col is None:
    print("No region column found; skipping region-level PCA.")
else:
    region_series = unit_meta[region_col].astype(str).fillna("unknown")
    keep_regions = region_series.value_counts()
    keep_regions = keep_regions[keep_regions >= 3].index

    if len(keep_regions) < 2:
        print("Not enough regions with >=3 units; skipping region-level PCA.")
    else:
        region_ids = [np.where(region_series.values == r)[0] for r in keep_regions]
        region_trials = np.stack([trials[:, ridx, :].mean(axis=1) for ridx in region_ids], axis=1)

        X_region = np.vstack([region_trials[i, :, in_task].mean(axis=1) for i in range(region_trials.shape[0])]).T
        X_region_z = zscore_rows(X_region)
        pca_region = PCA(n_components=min(6, X_region_z.shape[0], X_region_z.shape[1]))
        Xrp = pca_region.fit_transform(X_region_z.T).T
        print("Region PCA EVR (first 5):", np.round(pca_region.explained_variance_ratio_[:5], 4))

        plt.figure(figsize=(5, 4))
        for k, t in enumerate(trial_types):
            idx = t_type_ind[k]
            plt.scatter(Xrp[0, idx], Xrp[1, idx], s=28, alpha=0.8, color=pal[k], label=str(t))
        plt.xlabel("PC 1")
        plt.ylabel("PC 2")
        plt.legend(frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")
        sns.despine()
        plt.tight_layout()
        plt.show()


### Cell 13: Save Outputs and Close NWB Handle

**What this cell does**
- Saves trial tensor, PCA scores, trial table, and merged unit metadata to `extra_files`.
- Closes NWB file handle.

**How to use it**
- Run last.
- Use saved files for downstream stats, visualization, and reproducibility.


In [ ]:
# Save outputs
np.save(OUT_DIR / "allinone_trials_fr.npy", trials)
np.save(OUT_DIR / "allinone_trial_pca_scores.npy", Xp)
np.save(OUT_DIR / "allinone_traj_pca_scores.npy", Xa_p)
stim_df.to_csv(OUT_DIR / "allinone_trial_table.csv", index=False)
unit_meta.to_csv(OUT_DIR / "allinone_unit_meta.csv", index=False)

print("Saved:")
for p in [
    OUT_DIR / "allinone_trials_fr.npy",
    OUT_DIR / "allinone_trial_pca_scores.npy",
    OUT_DIR / "allinone_traj_pca_scores.npy",
    OUT_DIR / "allinone_trial_table.csv",
    OUT_DIR / "allinone_unit_meta.csv",
]:
    print(" ", p)

try:
    io.close()
except Exception:
    pass
